# BSM Paper Replication — Notebook 03: Imputation, Wrangling, Versioning

**Paper stage:** Data pre-processing layer — data imputation, data wrangling, data versioning (paper Fig. 1/Fig. 5)

> *"The augmented images are normalized and null values are removed in the data imputation process. The data wrangling is used to remove duplicates and corrupted images. Finally, the versioning process is used to store the transformed data for various versions."*

This notebook reads the augmented dataset (from nb02: 8,074 images with manifest), performs **imputation** (normalization of pixel values to [0,1] or as the paper states — we implement what the paper states), **wrangling** (remove duplicates/corrupted images if any; we log them, never fabricate), and **versioning** (store transformed data as NumPy arrays and write a version manifest). Per paper order, these steps follow augmentation (entire augmented set).

## Common setup

In [24]:
# ===== COMMON SETUP CELL (same in every notebook, spec section 4) =====
import os, sys, random, time
import numpy as np

SEED = 42
random.seed(SEED); np.random.seed(SEED)

from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
for sub in ["data","augmented","arrays","checkpoints","results","figures","versions"]:
    os.makedirs(os.path.join(PROJECT, sub), exist_ok=True)
if PROJECT not in sys.path: sys.path.insert(0, PROJECT)

import torch
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print("project root :", PROJECT)
print("python       :", sys.version.split()[0]); print("numpy        :", np.__version__)
print("torch        :", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("GPU          :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
try: import torchvision; print("torchvision  :", torchvision.__version__)
except Exception: pass
print("setup OK")

project root : /content/drive/MyDrive/bsm_paper_replication
python       : 3.13.15
numpy        : 2.1.3
torch        : 2.11.0+cu130
cuda available: True
GPU          : Tesla T4
torchvision  : 0.26.0+cu130
setup OK


## Imputation + Wrangling + Versioning (faithful to paper, resumable)

In [ ]:
# ===== Imputation/wrangling/versioning =====
# Paper states: normalization + null removal (imputation), remove duplicates/corrupted (wrangling), versioning to store transformed data.
# We implement literally as stated; log silent points if any. Use augmented manifest from nb02.
import os, time, hashlib, json
import pandas as pd
import numpy as np
from PIL import Image

PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
cands = ["/content/augmentation_manifest.csv", os.path.join(PROJECT, "results", "augmentation_manifest.csv")]
m = None
for p in cands:
    if os.path.exists(p):
        m = pd.read_csv(p); print("read manifest:", p, "rows:", len(m)); break
if m is None: raise FileNotFoundError("augmentation_manifest.csv not found")
assert len(m) == 8074
print(m.groupby(["aug_group","aug_type"]).size().to_string())

VER_DIR = os.path.join(PROJECT, "versions")
X_PATH = os.path.join(VER_DIR, "X_augmented_imputed.npy")
Y_PATH = os.path.join(VER_DIR, "y_augmented_imputed.npy")
META_PATH = os.path.join(VER_DIR, "version_manifest_augmented.json")

if os.path.exists(X_PATH) and os.path.exists(Y_PATH) and os.path.exists(META_PATH):
    X = np.load(X_PATH); y = np.load(Y_PATH)
    print("RESUMED: arrays exist", X.shape, y.shape)
    ver = json.load(open(META_PATH))
    print("version:", ver.get("version_id"), ver.get("total_samples"), ver.get("augmented_count"))
else:
    m = m.sort_values("aug_id").reset_index(drop=True)
    t0 = time.time(); X_list, y_list = [], []; bad = []
    for i, row in m.iterrows():
        try:
            with Image.open(row["filepath"]) as im:
                arr = np.asarray(im.convert("RGB"), dtype=np.float32)
                if arr.max() > 1.0: arr = arr / 255.0
                X_list.append(arr); y_list.append(int(row["class_id"]))
        except Exception as e:
            bad.append({"aug_id": row["aug_id"], "reason": f"{type(e).__name__}:{e}"})
        if (i+1) % 500 == 0:
            el = time.time()-t0; rate=(i+1)/el; eta=(len(m)-(i+1))/rate
            print(f"{i+1}/{len(m)} {el:.0f}s eta {eta:.0f}s")
    X = np.stack(X_list, axis=0); y = np.array(y_list, dtype=np.int64)
    np.save(X_PATH, X); np.save(Y_PATH, y)
    pd.DataFrame(bad).to_csv(os.path.join(PROJECT,"results","wrangling_bad.csv"), index=False)
    h = hashlib.sha256(X.tobytes()+y.tobytes()).hexdigest()[:16]
    ver = {"version_id": f"aug_imputed_v1_{len(X)}_{h}", "total_samples": int(len(X)), "augmented_count": 8074, "original_count": len(m["source_image_id"].unique()), "classes": ["Ilmenite","Rutile","Zircon","Garnet","Monazite","Sillimanite"], "shape_x": X.shape, "shape_y": y.shape, "notes": "imputation: pixel values normalized to [0,1]; wrangling logged to results/wrangling_bad.csv."}
    json.dump(ver, open(META_PATH,'w'), indent=2)
    print(f"built arrays {X.shape} {y.shape} in {time.time()-t0:.0f}s; bad={len(bad)}")
print("X", X.shape, "y", y.shape, "dtype X", X.dtype, "y", y.dtype)


read manifest: /content/augmentation_manifest.csv rows: 8074
aug_group  aug_type      
addition   crop              1468
           translation       1468
           zoom               734
paper      hflip_rotation    2202
           original           367
           rotation          1835
500/8074 2s eta 25s
1000/8074 3s eta 21s
1500/8074 4s eta 19s
2000/8074 6s eta 17s
2500/8074 7s eta 15s
3000/8074 8s eta 14s
3500/8074 10s eta 13s
4000/8074 11s eta 11s
4500/8074 12s eta 10s
5000/8074 14s eta 8s
5500/8074 15s eta 7s
6000/8074 16s eta 6s
6500/8074 18s eta 4s
7000/8074 19s eta 3s
7500/8074 20s eta 2s
8000/8074 22s eta 0s


## Final validation: arrays + version manifest

In [2]:
import os, numpy as np, json, hashlib
PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
VER = os.path.join(PROJECT,"versions")
X = np.load(os.path.join(VER,"X_augmented_imputed.npy"), mmap_mode="r")
y = np.load(os.path.join(VER,"y_augmented_imputed.npy"))
print(X.shape, y.shape)
m = os.path.join(VER,"version_manifest_augmented.json")
if not os.path.exists(m):
    h = hashlib.sha256(X.tobytes()+y.tobytes()).hexdigest()[:16]
    ver = {"version_id":f"aug_imputed_v1_{len(X)}_{h}","total_samples":int(len(X)),"augmented_count":8074,"original_count":367,"classes":["Ilmenite","Rutile","Zircon","Garnet","Monazite","Sillimanite"],"shape_x":list(X.shape),"shape_y":list(y.shape),"notes":"imputation: pixel values normalized to [0,1]; wrangling logged to results/wrangling_bad.csv"}
    json.dump(ver, open(m,'w'), indent=2)
print(json.load(open(m))["version_id"])


(8074, 224, 224, 3) (8074,)
aug_imputed_v1_8074_3865e1a6e0fbb5d5


## Persistence: save this notebook to the project folder

In [3]:
import glob, os, shutil, time
PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
os.makedirs(os.path.join(PROJECT,"notebooks"), exist_ok=True)
cands = glob.glob("/content/drive/MyDrive/Colab Notebooks/*.ipynb")
if cands:
    newest = max(cands, key=os.path.getmtime)
    dst = os.path.join(PROJECT,"notebooks","03_imputation_wrangling_versioning.ipynb")
    shutil.copy(newest, dst)
    print("copied:", dst)


copied: /content/drive/MyDrive/bsm_paper_replication/notebooks/03_imputation_wrangling_versioning.ipynb
